# 9.9 Prefill/Decode 分离与 KV 卸载

> 🕐 预估学习时间：40分钟

Prefill（提示预填充）计算密集、Decode（逐 token 生成）显存带宽密集。将两者分离到不同池（DistServe、Mooncake、vLLM PD）、并对 KV Cache 做分层卸载，是 2024–2026 推理集群的核心趋势。

本节涵盖：
- Prefill vs Decode 资源画像
- PD 分离调度
- KV Cache 卸载（GPU→CPU/远端）
- 端到端延迟与成本权衡


## 1. 为什么要分离 Prefill 和 Decode？

| 阶段 | 计算特点 | 瓶颈 | 批处理友好性 |
|------|---------|------|-------------|
| Prefill | 大矩阵乘，高算力 | FLOPs | 高（长提示可并行） |
| Decode | 小矩阵乘 + 读 KV | 显存带宽 | 需 continuous batching |

混部时，长提示 prefill 会打断 decode 批次，造成 TTFT/TPOT 抖动。


In [ ]:
import torch
import time
from dataclasses import dataclass, field

torch.manual_seed(42)


@dataclass
class Request:
    req_id: int
    prompt_len: int
    output_len: int
    generated: int = 0
    stage: str = 'prefill'  # prefill|decode|done
    ttft: float | None = None
    finish_t: float | None = None


def flops_prefill(prompt_len, d=4096, n_layers=32):
    # rough transformer FLOPs ~ 2 * n_layers * prompt_len * d^2 * c
    return 2 * n_layers * prompt_len * (d ** 2) * 6


def flops_decode_step(kv_len, d=4096, n_layers=32):
    return 2 * n_layers * 1 * (d ** 2) * 6 + n_layers * kv_len * d  # attn term


print('=== Prefill vs Decode Work Profile ===')
for p in [512, 2048, 8192]:
    print(f'prompt={p:>5}: prefill_FLOPs~{flops_prefill(p)/1e12:.2f}TF, '
          f'decode_step@kv={p}~{flops_decode_step(p)/1e9:.2f}GF')

print(f'\nKey: Prefill cost scales with prompt length; decode is many bandwidth-heavy steps.')


## 2. PD 分离调度模拟

- **Prefill Pool**：高算力 GPU，吃长提示
- **Decode Pool**：高带宽 / 更多并发槽位，专责生成
- 交接：prefill 完成后把 KV 传到 decode 池（或共享存储）


In [ ]:
from dataclasses import dataclass


@dataclass
class Request:
    req_id: int
    prompt_len: int
    output_len: int
    arrival: float = 0.0
    generated: int = 0
    stage: str = 'waiting'
    ttft: float | None = None
    finish_t: float | None = None


def simulate(disaggregated=False, horizon=5.0, dt=0.001):
    # Colocated: prefill interferes with decode. Disaggregated: parallel pools.
    specs = [(4096, 80), (512, 160), (2048, 40), (1024, 120)] * 4
    reqs = [Request(i, plen, olen, arrival=i * 0.01) for i, (plen, olen) in enumerate(specs)]

    prefill_tp = 80.0
    decode_tp = 4000.0
    now = 0.0
    prefill_q = []
    decode_q = []
    active_prefill = None
    prefill_remain = 0.0
    finished = []
    idx = 0

    while now < horizon and (
        idx < len(reqs) or prefill_q or decode_q or active_prefill
    ):
        while idx < len(reqs) and reqs[idx].arrival <= now:
            prefill_q.append(reqs[idx])
            idx += 1

        if active_prefill is None and prefill_q:
            active_prefill = prefill_q.pop(0)
            active_prefill.stage = 'prefill'
            prefill_remain = active_prefill.prompt_len / 512.0

        if active_prefill is not None:
            effective = prefill_tp if disaggregated else prefill_tp * 0.45
            prefill_remain -= effective * dt / 10.0
            if prefill_remain <= 0:
                active_prefill.ttft = now
                active_prefill.stage = 'decode'
                decode_q.append(active_prefill)
                active_prefill = None

        if decode_q:
            if disaggregated:
                rate = decode_tp
            else:
                rate = decode_tp * (0.25 if active_prefill is not None else 0.7)
            per = rate * dt / max(len(decode_q), 1)
            still = []
            for r in decode_q:
                r.generated += per
                if r.generated >= r.output_len:
                    r.stage = 'done'
                    r.finish_t = now
                    finished.append(r)
                else:
                    still.append(r)
            decode_q = still

        now += dt

    ttfts = [r.ttft for r in finished if r.ttft is not None]
    e2e = [r.finish_t for r in finished if r.finish_t is not None]
    return {
        'finished': len(finished),
        'avg_ttft': sum(ttfts) / len(ttfts) if ttfts else float('nan'),
        'avg_e2e': sum(e2e) / len(e2e) if e2e else float('nan'),
        'p95_ttft': sorted(ttfts)[int(0.95 * (len(ttfts) - 1))] if ttfts else float('nan'),
    }


print('=== Scheduling Comparison ===')
for name, flag in [('colocated-mixed', False), ('pd-disaggregated', True)]:
    r = simulate(disaggregated=flag)
    print(f"{name:<18} finished={r['finished']:<3} avg_ttft={r['avg_ttft']:.4f}s  "
          f"p95_ttft={r['p95_ttft']:.4f}s  avg_e2e={r['avg_e2e']:.4f}s")
print('\nKey: Disaggregating prefill/decode reduces interference and stabilizes TTFT/TPOT.')



## 3. KV Cache 分层卸载

显存不够时，将冷 KV 放到 CPU/NVMe/远端内存；命中时再换入。

权衡：卸载节省 GPU 显存 → 提高并发；换入增加 TPOT。适合多轮长会话中的历史前缀。


In [ ]:
class TieredKVCache:
    def __init__(self, gpu_blocks=16, cpu_blocks=64, block_tokens=16):
        self.block_tokens = block_tokens
        self.gpu = {}  # block_id -> tensor
        self.cpu = {}
        self.gpu_blocks = gpu_blocks
        self.cpu_blocks = cpu_blocks
        self.hits = 0
        self.miss_fetch = 0

    def _nbytes(self, n_blocks, d=64, layers=4):
        return n_blocks * self.block_tokens * d * layers * 2 * 2  # K,V fp16

    def put(self, session, tokens):
        n_blocks = (len(tokens) + self.block_tokens - 1) // self.block_tokens
        for b in range(n_blocks):
            key = (session, b)
            payload = torch.randn(self.block_tokens, 64)
            if len(self.gpu) < self.gpu_blocks:
                self.gpu[key] = payload
            elif len(self.cpu) < self.cpu_blocks:
                self.cpu[key] = payload
            else:
                # evict oldest cpu
                self.cpu.pop(next(iter(self.cpu)))
                self.cpu[key] = payload

    def get(self, session, block_id):
        key = (session, block_id)
        if key in self.gpu:
            self.hits += 1
            return self.gpu[key]
        if key in self.cpu:
            self.miss_fetch += 1
            # promote to gpu
            tensor = self.cpu.pop(key)
            if len(self.gpu) >= self.gpu_blocks:
                # demote one gpu block
                old_k = next(iter(self.gpu))
                self.cpu[old_k] = self.gpu.pop(old_k)
            self.gpu[key] = tensor
            return tensor
        return None


cache = TieredKVCache()
for s in range(6):
    cache.put(f'sess-{s}', list(range(80 + 10 * s)))

for s in range(6):
    cache.get(f'sess-{s}', 0)
    cache.get(f'sess-{s}', 1)

print('=== Tiered KV Cache ===')
print(f'GPU blocks used: {len(cache.gpu)} / {cache.gpu_blocks}')
print(f'CPU blocks used: {len(cache.cpu)} / {cache.cpu_blocks}')
print(f'GPU hits: {cache.hits}, CPU promotions: {cache.miss_fetch}')
print(f'Approx GPU KV bytes: {cache._nbytes(len(cache.gpu))/1024:.1f} KB (toy dims)')
print(f'\nKey: Tiered KV raises concurrency; promotions must be overlapped with compute to hide latency.')


## 课后思考题

1. 什么样的流量形态（短提示长生成 / 长提示短生成）从 PD 分离中收益最大？
2. KV 跨节点传输成为瓶颈时，有哪些缓解（量化 KV、前缀缓存、同机共享内存）？
3. PD 分离与 Continuous Batching、前缀缓存如何协同设计？
4. 卸载到 CPU 后，如何设定换入换出策略避免抖动？

---
> 本节涵盖了9.9 Prefill/Decode 分离与 KV 卸载的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
